In [2]:
##CUDA


!nvidia-smi

Wed Oct  7 16:31:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [5]:
#1st data set
%%writefile gpu_dataset.cu

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

__global__ void sumKernel(const float* data, double* result, long long N) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N) {
        atomicAdd(result, (double)data[idx]);
    }
}

int main() {

    const long long N = 100000;   // 1 Lakh
    const int threadsPerBlock = 128;

    size_t dataSize = N * sizeof(float);

    // Host memory
    float* h_data = new float[N];

    // Generate dataset
    unsigned int seed = 42;

    for (long long i = 0; i < N; i++) {
        seed = 1664525 * seed + 1013904223;
        h_data[i] = 1.0f + (seed % 9901) / 100.0f;
    }

    // Device memory
    float* d_data;
    double* d_result;

    cudaMalloc(&d_data, dataSize);
    cudaMalloc(&d_result, sizeof(double));

    double zero = 0.0;

    cudaMemcpy(d_data, h_data, dataSize, cudaMemcpyHostToDevice);
    cudaMemcpy(d_result, &zero, sizeof(double), cudaMemcpyHostToDevice);

    // Number of blocks
    int blocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    // GPU timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    sumKernel<<<blocks, threadsPerBlock>>>(d_data, d_result, N);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpuTime;
    cudaEventElapsedTime(&gpuTime, start, stop);

    // Copy result back
    double sum;

    cudaMemcpy(&sum, d_result, sizeof(double),
               cudaMemcpyDeviceToHost);

    double average = sum / N;

    cout << "Dataset Size: " << N << endl;
    cout << "Threads per Block: " << threadsPerBlock << endl;
    cout << "Blocks: " << blocks << endl;
    cout << "Sum: " << sum << endl;
    cout << "Average: " << average << endl;
    cout << "GPU Execution Time: "
         << gpuTime / 1000.0 << " seconds" << endl;

    // Cleanup
    cudaFree(d_data);
    cudaFree(d_result);
    delete[] h_data;

    return 0;
}

Writing gpu_dataset.cu


In [6]:
!nvcc -O2 gpu_dataset.cu -o gpu_dataset

In [7]:
!./gpu_dataset

Dataset Size: 100000
Threads per Block: 128
Blocks: 782
Sum: 5.06744e+06
Average: 50.6744
GPU Execution Time: 0.00044464 seconds


In [8]:
%%writefile gpu_dataset.cu

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

__global__ void sumKernel(const float* data, double* result, long long N) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N) {
        atomicAdd(result, (double)data[idx]);
    }
}

int main() {

    const long long N = 100000;   // 1 Lakh
    const int threadsPerBlock = 256;

    size_t dataSize = N * sizeof(float);

    // Host memory
    float* h_data = new float[N];

    // Generate dataset
    unsigned int seed = 42;

    for (long long i = 0; i < N; i++) {
        seed = 1664525 * seed + 1013904223;
        h_data[i] = 1.0f + (seed % 9901) / 100.0f;
    }

    // Device memory
    float* d_data;
    double* d_result;

    cudaMalloc(&d_data, dataSize);
    cudaMalloc(&d_result, sizeof(double));

    double zero = 0.0;

    cudaMemcpy(d_data, h_data, dataSize, cudaMemcpyHostToDevice);
    cudaMemcpy(d_result, &zero, sizeof(double), cudaMemcpyHostToDevice);

    // Number of blocks
    int blocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    // GPU timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    sumKernel<<<blocks, threadsPerBlock>>>(d_data, d_result, N);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpuTime;
    cudaEventElapsedTime(&gpuTime, start, stop);

    // Copy result back
    double sum;

    cudaMemcpy(&sum, d_result, sizeof(double),
               cudaMemcpyDeviceToHost);

    double average = sum / N;

    cout << "Dataset Size: " << N << endl;
    cout << "Threads per Block: " << threadsPerBlock << endl;
    cout << "Blocks: " << blocks << endl;
    cout << "Sum: " << sum << endl;
    cout << "Average: " << average << endl;
    cout << "GPU Execution Time: "
         << gpuTime / 1000.0 << " seconds" << endl;

    // Cleanup
    cudaFree(d_data);
    cudaFree(d_result);
    delete[] h_data;

    return 0;
}

Overwriting gpu_dataset.cu


In [9]:
!nvcc -O2 gpu_dataset.cu -o gpu_dataset
!./gpu_dataset

Dataset Size: 100000
Threads per Block: 256
Blocks: 391
Sum: 5.06744e+06
Average: 50.6744
GPU Execution Time: 0.0004512 seconds


In [10]:
%%writefile gpu_dataset.cu

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

__global__ void sumKernel(const float* data, double* result, long long N) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N) {
        atomicAdd(result, (double)data[idx]);
    }
}

int main() {

    const long long N = 100000;   // 1 Lakh
    const int threadsPerBlock = 512;

    size_t dataSize = N * sizeof(float);

    // Host memory
    float* h_data = new float[N];

    // Generate dataset
    unsigned int seed = 42;

    for (long long i = 0; i < N; i++) {
        seed = 1664525 * seed + 1013904223;
        h_data[i] = 1.0f + (seed % 9901) / 100.0f;
    }

    // Device memory
    float* d_data;
    double* d_result;

    cudaMalloc(&d_data, dataSize);
    cudaMalloc(&d_result, sizeof(double));

    double zero = 0.0;

    cudaMemcpy(d_data, h_data, dataSize, cudaMemcpyHostToDevice);
    cudaMemcpy(d_result, &zero, sizeof(double), cudaMemcpyHostToDevice);

    // Number of blocks
    int blocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    // GPU timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    sumKernel<<<blocks, threadsPerBlock>>>(d_data, d_result, N);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpuTime;
    cudaEventElapsedTime(&gpuTime, start, stop);

    // Copy result back
    double sum;

    cudaMemcpy(&sum, d_result, sizeof(double),
               cudaMemcpyDeviceToHost);

    double average = sum / N;

    cout << "Dataset Size: " << N << endl;
    cout << "Threads per Block: " << threadsPerBlock << endl;
    cout << "Blocks: " << blocks << endl;
    cout << "Sum: " << sum << endl;
    cout << "Average: " << average << endl;
    cout << "GPU Execution Time: "
         << gpuTime / 1000.0 << " seconds" << endl;

    // Cleanup
    cudaFree(d_data);
    cudaFree(d_result);
    delete[] h_data;

    return 0;
}

Overwriting gpu_dataset.cu


In [11]:
!nvcc -O2 gpu_dataset.cu -o gpu_dataset
!./gpu_dataset

Dataset Size: 100000
Threads per Block: 512
Blocks: 196
Sum: 5.06744e+06
Average: 50.6744
GPU Execution Time: 0.00044192 seconds


In [12]:
# 2nd datset 500000
%%writefile gpu_dataset.cu

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

__global__ void sumKernel(const float* data, double* result, long long N) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N) {
        atomicAdd(result, (double)data[idx]);
    }
}

int main() {

    const long long N = 500000;   // 5 Lakh
    const int threadsPerBlock = 128;

    size_t dataSize = N * sizeof(float);

    // Host memory
    float* h_data = new float[N];

    // Generate dataset
    unsigned int seed = 42;

    for (long long i = 0; i < N; i++) {
        seed = 1664525 * seed + 1013904223;
        h_data[i] = 1.0f + (seed % 9901) / 100.0f;
    }

    // Device memory
    float* d_data;
    double* d_result;

    cudaMalloc(&d_data, dataSize);
    cudaMalloc(&d_result, sizeof(double));

    double zero = 0.0;

    cudaMemcpy(d_data, h_data, dataSize, cudaMemcpyHostToDevice);
    cudaMemcpy(d_result, &zero, sizeof(double), cudaMemcpyHostToDevice);

    // Number of blocks
    int blocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    // GPU timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    sumKernel<<<blocks, threadsPerBlock>>>(d_data, d_result, N);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpuTime;
    cudaEventElapsedTime(&gpuTime, start, stop);

    // Copy result back
    double sum;

    cudaMemcpy(&sum, d_result, sizeof(double),
               cudaMemcpyDeviceToHost);

    double average = sum / N;

    cout << "Dataset Size: " << N << endl;
    cout << "Threads per Block: " << threadsPerBlock << endl;
    cout << "Blocks: " << blocks << endl;
    cout << "Sum: " << sum << endl;
    cout << "Average: " << average << endl;
    cout << "GPU Execution Time: "
         << gpuTime / 1000.0 << " seconds" << endl;

    // Cleanup
    cudaFree(d_data);
    cudaFree(d_result);
    delete[] h_data;

    return 0;
}

Overwriting gpu_dataset.cu


In [13]:
!nvcc -O2 gpu_dataset.cu -o gpu_dataset
!./gpu_dataset

Dataset Size: 500000
Threads per Block: 128
Blocks: 3907
Sum: 2.52629e+07
Average: 50.5257
GPU Execution Time: 0.00186368 seconds


In [14]:
# 2nd datset 500000
%%writefile gpu_dataset.cu

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

__global__ void sumKernel(const float* data, double* result, long long N) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N) {
        atomicAdd(result, (double)data[idx]);
    }
}

int main() {

    const long long N = 500000;   // 5 Lakh
    const int threadsPerBlock = 256;

    size_t dataSize = N * sizeof(float);

    // Host memory
    float* h_data = new float[N];

    // Generate dataset
    unsigned int seed = 42;

    for (long long i = 0; i < N; i++) {
        seed = 1664525 * seed + 1013904223;
        h_data[i] = 1.0f + (seed % 9901) / 100.0f;
    }

    // Device memory
    float* d_data;
    double* d_result;

    cudaMalloc(&d_data, dataSize);
    cudaMalloc(&d_result, sizeof(double));

    double zero = 0.0;

    cudaMemcpy(d_data, h_data, dataSize, cudaMemcpyHostToDevice);
    cudaMemcpy(d_result, &zero, sizeof(double), cudaMemcpyHostToDevice);

    // Number of blocks
    int blocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    // GPU timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    sumKernel<<<blocks, threadsPerBlock>>>(d_data, d_result, N);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpuTime;
    cudaEventElapsedTime(&gpuTime, start, stop);

    // Copy result back
    double sum;

    cudaMemcpy(&sum, d_result, sizeof(double),
               cudaMemcpyDeviceToHost);

    double average = sum / N;

    cout << "Dataset Size: " << N << endl;
    cout << "Threads per Block: " << threadsPerBlock << endl;
    cout << "Blocks: " << blocks << endl;
    cout << "Sum: " << sum << endl;
    cout << "Average: " << average << endl;
    cout << "GPU Execution Time: "
         << gpuTime / 1000.0 << " seconds" << endl;

    // Cleanup
    cudaFree(d_data);
    cudaFree(d_result);
    delete[] h_data;

    return 0;
}

Overwriting gpu_dataset.cu


In [15]:
!nvcc -O2 gpu_dataset.cu -o gpu_dataset
!./gpu_dataset

Dataset Size: 500000
Threads per Block: 256
Blocks: 1954
Sum: 2.52629e+07
Average: 50.5257
GPU Execution Time: 0.00182208 seconds


In [19]:
# 2nd datset 500000
%%writefile gpu_dataset.cu

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

__global__ void sumKernel(const float* data, double* result, long long N) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N) {
        atomicAdd(result, (double)data[idx]);
    }
}

int main() {

    const long long N = 500000;   // 5 Lakh
    const int threadsPerBlock = 512;

    size_t dataSize = N * sizeof(float);

    // Host memory
    float* h_data = new float[N];

    // Generate dataset
    unsigned int seed = 42;

    for (long long i = 0; i < N; i++) {
        seed = 1664525 * seed + 1013904223;
        h_data[i] = 1.0f + (seed % 9901) / 100.0f;
    }

    // Device memory
    float* d_data;
    double* d_result;

    cudaMalloc(&d_data, dataSize);
    cudaMalloc(&d_result, sizeof(double));

    double zero = 0.0;

    cudaMemcpy(d_data, h_data, dataSize, cudaMemcpyHostToDevice);
    cudaMemcpy(d_result, &zero, sizeof(double), cudaMemcpyHostToDevice);

    // Number of blocks
    int blocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    // GPU timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    sumKernel<<<blocks, threadsPerBlock>>>(d_data, d_result, N);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpuTime;
    cudaEventElapsedTime(&gpuTime, start, stop);

    // Copy result back
    double sum;

    cudaMemcpy(&sum, d_result, sizeof(double),
               cudaMemcpyDeviceToHost);

    double average = sum / N;

    cout << "Dataset Size: " << N << endl;
    cout << "Threads per Block: " << threadsPerBlock << endl;
    cout << "Blocks: " << blocks << endl;
    cout << "Sum: " << sum << endl;
    cout << "Average: " << average << endl;
    cout << "GPU Execution Time: "
         << gpuTime / 1000.0 << " seconds" << endl;

    // Cleanup
    cudaFree(d_data);
    cudaFree(d_result);
    delete[] h_data;

    return 0;
}

Overwriting gpu_dataset.cu


In [20]:
!nvcc -O2 gpu_dataset.cu -o gpu_dataset
!./gpu_dataset

Dataset Size: 500000
Threads per Block: 512
Blocks: 977
Sum: 2.52629e+07
Average: 50.5257
GPU Execution Time: 0.00187805 seconds


In [21]:
#3rd datset
%%writefile gpu_dataset.cu

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

__global__ void sumKernel(const float* data, double* result, long long N) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N) {
        atomicAdd(result, (double)data[idx]);
    }
}

int main() {

    const long long N = 1000000;   // 10 Lakh
    const int threadsPerBlock = 128;

    size_t dataSize = N * sizeof(float);

    // Host memory
    float* h_data = new float[N];

    // Generate dataset
    unsigned int seed = 42;

    for (long long i = 0; i < N; i++) {
        seed = 1664525 * seed + 1013904223;
        h_data[i] = 1.0f + (seed % 9901) / 100.0f;
    }

    // Device memory
    float* d_data;
    double* d_result;

    cudaMalloc(&d_data, dataSize);
    cudaMalloc(&d_result, sizeof(double));

    double zero = 0.0;

    cudaMemcpy(d_data, h_data, dataSize, cudaMemcpyHostToDevice);
    cudaMemcpy(d_result, &zero, sizeof(double), cudaMemcpyHostToDevice);

    // Number of blocks
    int blocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    // GPU timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    sumKernel<<<blocks, threadsPerBlock>>>(d_data, d_result, N);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpuTime;
    cudaEventElapsedTime(&gpuTime, start, stop);

    // Copy result back
    double sum;

    cudaMemcpy(&sum, d_result, sizeof(double),
               cudaMemcpyDeviceToHost);

    double average = sum / N;

    cout << "Dataset Size: " << N << endl;
    cout << "Threads per Block: " << threadsPerBlock << endl;
    cout << "Blocks: " << blocks << endl;
    cout << "Sum: " << sum << endl;
    cout << "Average: " << average << endl;
    cout << "GPU Execution Time: "
         << gpuTime / 1000.0 << " seconds" << endl;

    // Cleanup
    cudaFree(d_data);
    cudaFree(d_result);
    delete[] h_data;

    return 0;
}

Overwriting gpu_dataset.cu


In [22]:
!nvcc -O2 gpu_dataset.cu -o gpu_dataset
!./gpu_dataset

Dataset Size: 1000000
Threads per Block: 128
Blocks: 7813
Sum: 5.05025e+07
Average: 50.5025
GPU Execution Time: 0.00360448 seconds


In [23]:
#3rd datset
%%writefile gpu_dataset.cu

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

__global__ void sumKernel(const float* data, double* result, long long N) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N) {
        atomicAdd(result, (double)data[idx]);
    }
}

int main() {

    const long long N = 1000000;   // 10 Lakh
    const int threadsPerBlock = 256;

    size_t dataSize = N * sizeof(float);

    // Host memory
    float* h_data = new float[N];

    // Generate dataset
    unsigned int seed = 42;

    for (long long i = 0; i < N; i++) {
        seed = 1664525 * seed + 1013904223;
        h_data[i] = 1.0f + (seed % 9901) / 100.0f;
    }

    // Device memory
    float* d_data;
    double* d_result;

    cudaMalloc(&d_data, dataSize);
    cudaMalloc(&d_result, sizeof(double));

    double zero = 0.0;

    cudaMemcpy(d_data, h_data, dataSize, cudaMemcpyHostToDevice);
    cudaMemcpy(d_result, &zero, sizeof(double), cudaMemcpyHostToDevice);

    // Number of blocks
    int blocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    // GPU timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    sumKernel<<<blocks, threadsPerBlock>>>(d_data, d_result, N);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpuTime;
    cudaEventElapsedTime(&gpuTime, start, stop);

    // Copy result back
    double sum;

    cudaMemcpy(&sum, d_result, sizeof(double),
               cudaMemcpyDeviceToHost);

    double average = sum / N;

    cout << "Dataset Size: " << N << endl;
    cout << "Threads per Block: " << threadsPerBlock << endl;
    cout << "Blocks: " << blocks << endl;
    cout << "Sum: " << sum << endl;
    cout << "Average: " << average << endl;
    cout << "GPU Execution Time: "
         << gpuTime / 1000.0 << " seconds" << endl;

    // Cleanup
    cudaFree(d_data);
    cudaFree(d_result);
    delete[] h_data;

    return 0;
}

Overwriting gpu_dataset.cu


In [24]:
!nvcc -O2 gpu_dataset.cu -o gpu_dataset
!./gpu_dataset

Dataset Size: 1000000
Threads per Block: 256
Blocks: 3907
Sum: 5.05025e+07
Average: 50.5025
GPU Execution Time: 0.00361158 seconds


In [25]:
#3rd datset
%%writefile gpu_dataset.cu

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

__global__ void sumKernel(const float* data, double* result, long long N) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N) {
        atomicAdd(result, (double)data[idx]);
    }
}

int main() {

    const long long N = 1000000;   // 10 Lakh
    const int threadsPerBlock = 512;

    size_t dataSize = N * sizeof(float);

    // Host memory
    float* h_data = new float[N];

    // Generate dataset
    unsigned int seed = 42;

    for (long long i = 0; i < N; i++) {
        seed = 1664525 * seed + 1013904223;
        h_data[i] = 1.0f + (seed % 9901) / 100.0f;
    }

    // Device memory
    float* d_data;
    double* d_result;

    cudaMalloc(&d_data, dataSize);
    cudaMalloc(&d_result, sizeof(double));

    double zero = 0.0;

    cudaMemcpy(d_data, h_data, dataSize, cudaMemcpyHostToDevice);
    cudaMemcpy(d_result, &zero, sizeof(double), cudaMemcpyHostToDevice);

    // Number of blocks
    int blocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    // GPU timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    sumKernel<<<blocks, threadsPerBlock>>>(d_data, d_result, N);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpuTime;
    cudaEventElapsedTime(&gpuTime, start, stop);

    // Copy result back
    double sum;

    cudaMemcpy(&sum, d_result, sizeof(double),
               cudaMemcpyDeviceToHost);

    double average = sum / N;

    cout << "Dataset Size: " << N << endl;
    cout << "Threads per Block: " << threadsPerBlock << endl;
    cout << "Blocks: " << blocks << endl;
    cout << "Sum: " << sum << endl;
    cout << "Average: " << average << endl;
    cout << "GPU Execution Time: "
         << gpuTime / 1000.0 << " seconds" << endl;

    // Cleanup
    cudaFree(d_data);
    cudaFree(d_result);
    delete[] h_data;

    return 0;
}

Overwriting gpu_dataset.cu


In [26]:
!nvcc -O2 gpu_dataset.cu -o gpu_dataset
!./gpu_dataset

Dataset Size: 1000000
Threads per Block: 512
Blocks: 1954
Sum: 5.05025e+07
Average: 50.5025
GPU Execution Time: 0.00323046 seconds


In [27]:
#4th datset
%%writefile gpu_dataset.cu

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

__global__ void sumKernel(const float* data, double* result, long long N) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N) {
        atomicAdd(result, (double)data[idx]);
    }
}

int main() {

    const long long N = 2000000;   // 20 Lakh
    const int threadsPerBlock = 128;

    size_t dataSize = N * sizeof(float);

    // Host memory
    float* h_data = new float[N];

    // Generate dataset
    unsigned int seed = 42;

    for (long long i = 0; i < N; i++) {
        seed = 1664525 * seed + 1013904223;
        h_data[i] = 1.0f + (seed % 9901) / 100.0f;
    }

    // Device memory
    float* d_data;
    double* d_result;

    cudaMalloc(&d_data, dataSize);
    cudaMalloc(&d_result, sizeof(double));

    double zero = 0.0;

    cudaMemcpy(d_data, h_data, dataSize, cudaMemcpyHostToDevice);
    cudaMemcpy(d_result, &zero, sizeof(double), cudaMemcpyHostToDevice);

    // Number of blocks
    int blocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    // GPU timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    sumKernel<<<blocks, threadsPerBlock>>>(d_data, d_result, N);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpuTime;
    cudaEventElapsedTime(&gpuTime, start, stop);

    // Copy result back
    double sum;

    cudaMemcpy(&sum, d_result, sizeof(double),
               cudaMemcpyDeviceToHost);

    double average = sum / N;

    cout << "Dataset Size: " << N << endl;
    cout << "Threads per Block: " << threadsPerBlock << endl;
    cout << "Blocks: " << blocks << endl;
    cout << "Sum: " << sum << endl;
    cout << "Average: " << average << endl;
    cout << "GPU Execution Time: "
         << gpuTime / 1000.0 << " seconds" << endl;

    // Cleanup
    cudaFree(d_data);
    cudaFree(d_result);
    delete[] h_data;

    return 0;
}

Overwriting gpu_dataset.cu


In [28]:
!nvcc -O2 gpu_dataset.cu -o gpu_dataset
!./gpu_dataset

Dataset Size: 2000000
Threads per Block: 128
Blocks: 15625
Sum: 1.00983e+08
Average: 50.4913
GPU Execution Time: 0.00711504 seconds


In [29]:
#4th datset
%%writefile gpu_dataset.cu

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

__global__ void sumKernel(const float* data, double* result, long long N) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N) {
        atomicAdd(result, (double)data[idx]);
    }
}

int main() {

    const long long N = 2000000;   // 20 Lakh
    const int threadsPerBlock = 256;

    size_t dataSize = N * sizeof(float);

    // Host memory
    float* h_data = new float[N];

    // Generate dataset
    unsigned int seed = 42;

    for (long long i = 0; i < N; i++) {
        seed = 1664525 * seed + 1013904223;
        h_data[i] = 1.0f + (seed % 9901) / 100.0f;
    }

    // Device memory
    float* d_data;
    double* d_result;

    cudaMalloc(&d_data, dataSize);
    cudaMalloc(&d_result, sizeof(double));

    double zero = 0.0;

    cudaMemcpy(d_data, h_data, dataSize, cudaMemcpyHostToDevice);
    cudaMemcpy(d_result, &zero, sizeof(double), cudaMemcpyHostToDevice);

    // Number of blocks
    int blocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    // GPU timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    sumKernel<<<blocks, threadsPerBlock>>>(d_data, d_result, N);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpuTime;
    cudaEventElapsedTime(&gpuTime, start, stop);

    // Copy result back
    double sum;

    cudaMemcpy(&sum, d_result, sizeof(double),
               cudaMemcpyDeviceToHost);

    double average = sum / N;

    cout << "Dataset Size: " << N << endl;
    cout << "Threads per Block: " << threadsPerBlock << endl;
    cout << "Blocks: " << blocks << endl;
    cout << "Sum: " << sum << endl;
    cout << "Average: " << average << endl;
    cout << "GPU Execution Time: "
         << gpuTime / 1000.0 << " seconds" << endl;

    // Cleanup
    cudaFree(d_data);
    cudaFree(d_result);
    delete[] h_data;

    return 0;
}

Overwriting gpu_dataset.cu


In [30]:
!nvcc -O2 gpu_dataset.cu -o gpu_dataset
!./gpu_dataset

Dataset Size: 2000000
Threads per Block: 256
Blocks: 7813
Sum: 1.00983e+08
Average: 50.4913
GPU Execution Time: 0.00711136 seconds


In [31]:
#4th datset
%%writefile gpu_dataset.cu

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

__global__ void sumKernel(const float* data, double* result, long long N) {

    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < N) {
        atomicAdd(result, (double)data[idx]);
    }
}

int main() {

    const long long N = 2000000;   // 20 Lakh
    const int threadsPerBlock = 512;

    size_t dataSize = N * sizeof(float);

    // Host memory
    float* h_data = new float[N];

    // Generate dataset
    unsigned int seed = 42;

    for (long long i = 0; i < N; i++) {
        seed = 1664525 * seed + 1013904223;
        h_data[i] = 1.0f + (seed % 9901) / 100.0f;
    }

    // Device memory
    float* d_data;
    double* d_result;

    cudaMalloc(&d_data, dataSize);
    cudaMalloc(&d_result, sizeof(double));

    double zero = 0.0;

    cudaMemcpy(d_data, h_data, dataSize, cudaMemcpyHostToDevice);
    cudaMemcpy(d_result, &zero, sizeof(double), cudaMemcpyHostToDevice);

    // Number of blocks
    int blocks = (N + threadsPerBlock - 1) / threadsPerBlock;

    // GPU timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    sumKernel<<<blocks, threadsPerBlock>>>(d_data, d_result, N);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpuTime;
    cudaEventElapsedTime(&gpuTime, start, stop);

    // Copy result back
    double sum;

    cudaMemcpy(&sum, d_result, sizeof(double),
               cudaMemcpyDeviceToHost);

    double average = sum / N;

    cout << "Dataset Size: " << N << endl;
    cout << "Threads per Block: " << threadsPerBlock << endl;
    cout << "Blocks: " << blocks << endl;
    cout << "Sum: " << sum << endl;
    cout << "Average: " << average << endl;
    cout << "GPU Execution Time: "
         << gpuTime / 1000.0 << " seconds" << endl;

    // Cleanup
    cudaFree(d_data);
    cudaFree(d_result);
    delete[] h_data;

    return 0;
}

Overwriting gpu_dataset.cu


In [32]:
!nvcc -O2 gpu_dataset.cu -o gpu_dataset
!./gpu_dataset

Dataset Size: 2000000
Threads per Block: 512
Blocks: 3907
Sum: 1.00983e+08
Average: 50.4913
GPU Execution Time: 0.00711926 seconds
